# Modélisation et évaluation : Telco Churn

Ce notebook utilise les mêmes fonctions de `src/` que l'application et l'API :
1. Préparation  2. Modèles et hyperparamètres  3. Évaluation (métriques, ROC, PR, confusion, seuil)  4. Appel de l'API (optionnel)

> Lancer depuis la racine du projet avec le venv actif (`jupyter notebook`).

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd

from src import data as D, preprocess as P, models as M, plots

## 1. Données et préparation

In [2]:
df = D.generic_clean(D.load_data())                      # TotalCharges -> numérique, customerID supprimé
df["Churn"] = P.encode_target(df["Churn"], positive_label="Yes")

prep = P.prepare(df, target="Churn", test_size=0.2, imputer="median", scale=True)
X_train, X_test, y_train, y_test = prep["X_train"], prep["X_test"], prep["y_train"], prep["y_test"]
print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Taux de churn  train = %.3f  test = %.3f" % (y_train.mean(), y_test.mean()))
print("Colonnes après encodage :", len(prep["feature_names"]))

Train : (5634, 21) | Test : (1409, 21)
Taux de churn  train = 0.265  test = 0.265
Colonnes après encodage : 28


## 2. Modèles et hyperparamètres

`RandomizedSearchCV` (score AUC, validation croisée). Le prétraitement est ré-ajusté sur chaque pli : pas de fuite de données.
Déséquilibre : `class_weight="balanced"` (`scale_pos_weight` pour XGBoost).

In [3]:
pipelines, best_params, cv_auc = {}, {}, {}
for name in M.MODEL_NAMES:
    pipelines[name], best_params[name], cv_auc[name] = M.train_model(
        name, prep["preprocessor"], X_train, y_train, tune=True, n_iter=12, cv=3)
    print(f"{name:<20s} AUC (CV) = {cv_auc[name]:.3f}  {best_params[name]}")

Logistic Regression  AUC (CV) = 0.846  {'C': 10}
Decision Tree        AUC (CV) = 0.832  {'min_samples_leaf': 50, 'max_depth': 5}
KNN                  AUC (CV) = 0.833  {'weights': 'uniform', 'n_neighbors': 41}
Random Forest        AUC (CV) = 0.848  {'n_estimators': 200, 'min_samples_leaf': 20, 'max_depth': 8}
XGBoost              AUC (CV) = 0.850  {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}


## 3. Évaluation

In [4]:
scores = M.score_models(pipelines, X_test)               # probabilités de la classe positive
results = M.evaluate_scores(scores, y_test, threshold=0.5)
results

,AUC-ROC,PR-AUC,Précision,Rappel,F1,Accuracy,Gini,KS
XGBoost,0.8459,0.6584,0.5128,0.8048,0.6264,0.7452,0.6918,0.5390
Random Forest,0.8452,0.6572,0.5219,0.7968,0.6307,0.7523,0.6903,0.5516
Logistic Regression,0.8409,0.6273,0.5052,0.7807,0.6134,0.7388,0.6818,0.5174
Decision Tree,0.8361,0.6220,0.5244,0.8048,0.6350,0.7544,0.6722,0.5490
KNN,0.8304,0.6226,0.6068,0.5695,0.5876,0.7878,0.6607,0.4989


In [5]:
plots.plot_roc(scores, y_test).show()
plots.plot_pr(scores, y_test).show()

In [6]:
best = results.index[0]
t_opt = M.best_threshold_f1(y_test, scores[best])
print(f"Meilleur modèle : {best} | seuil optimal (F1) = {t_opt:.2f}")
print("Au seuil 0.50 :", M.confusion_counts(y_test, scores[best], 0.5))
print(f"Au seuil {t_opt:.2f}:", M.confusion_counts(y_test, scores[best], t_opt))
plots.plot_confusion(y_test, scores[best], t_opt).show()
plots.plot_threshold_curve(y_test, scores[best], t_opt).show()

Meilleur modèle : XGBoost | seuil optimal (F1) = 0.60
Au seuil 0.50 : {'VN': 749, 'FP': 286, 'FN': 73, 'VP': 301}
Au seuil 0.60: {'VN': 829, 'FP': 206, 'FN': 99, 'VP': 275}


**Lecture :** baisser le seuil augmente le rappel (moins de départs ratés) mais réduit la précision (plus de fausses alertes).
Le bon seuil dépend du coût d'une action de rétention par rapport au coût d'un client perdu.

In [7]:
# Compromis performance / interprétabilité
lr = results.loc["Logistic Regression", "AUC-ROC"]
print(f"AUC meilleur modèle ({best}) = {results.loc[best, 'AUC-ROC']:.3f} | régression logistique = {lr:.3f} | écart = {results.loc[best, 'AUC-ROC'] - lr:+.3f}")

AUC meilleur modèle (XGBoost) = 0.846 | régression logistique = 0.841 | écart = +0.005


## 4. (Optionnel) Utiliser l'API

Démarrer l'API dans un terminal : `uvicorn api.main:app --port 8000`, puis exécuter cette cellule :
`POST /train` lance l'entraînement en tâche de fond, `GET /jobs/{id}` donne l'avancement et le résultat.

In [8]:
import json, time, requests

API = "http://localhost:8000"
try:
    cfg = {"target": "Churn", "positive_label": 1, "tune": False, "models": ["Logistic Regression", "XGBoost"]}
    records = json.loads(df.to_json(orient="records"))
    job = requests.post(f"{API}/train", json={"data": records, **cfg}, timeout=120).json()
    while True:
        j = requests.get(f"{API}/jobs/{job['job_id']}").json()
        print(j["status"], f"{j['progress']:.0%}", j["message"])
        if j["status"] != "running":
            break
        time.sleep(1)
    sid = j["result"]["session_id"]
    print(requests.get(f"{API}/sessions/{sid}/metrics", params={"threshold": 0.5}).json()["metrics"])
except requests.RequestException as e:
    print("API non démarrée :", type(e).__name__)

API non démarrée : ConnectionError


In [9]:
print("=== RÉSUMÉ À COPIER ===")
print("Train/Test :", X_train.shape, X_test.shape,
      "| churn train/test : %.3f / %.3f" % (y_train.mean(), y_test.mean()))
print("\nMétriques de test (seuil 0.5) :")
print(results.to_string())
print("\nHyperparamètres retenus :")
for k, v in best_params.items():
    print(" ", k, v, "| AUC CV =", round(cv_auc[k], 3))
print("\nMeilleur modèle :", best, "| seuil optimal F1 :", round(t_opt, 2))
print("Confusion seuil 0.5 :", M.confusion_counts(y_test, scores[best], 0.5))
print("Confusion seuil opt :", M.confusion_counts(y_test, scores[best], t_opt))
print("Métriques au seuil opt :",
      {k: round(v, 3) for k, v in M.compute_metrics(y_test, scores[best], t_opt).items()})
print("\nTaux de churn par modalité :")
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport",
            "OnlineSecurity", "SeniorCitizen", "gender"]:
    print(" ", col, df.groupby(col)["Churn"].mean().round(3).to_dict())
print("corr(tenure, TotalCharges) =", round(df["tenure"].corr(df["TotalCharges"]), 3))
print("TotalCharges manquants :", int(df["TotalCharges"].isna().sum()))

=== RÉSUMÉ À COPIER ===
Train/Test : (5634, 21) (1409, 21) | churn train/test : 0.265 / 0.265

Métriques de test (seuil 0.5) :
                     AUC-ROC  PR-AUC  Précision  Rappel      F1  Accuracy    Gini      KS
XGBoost               0.8459  0.6584     0.5128  0.8048  0.6264    0.7452  0.6918  0.5390
Random Forest         0.8452  0.6572     0.5219  0.7968  0.6307    0.7523  0.6903  0.5516
Logistic Regression   0.8409  0.6273     0.5052  0.7807  0.6134    0.7388  0.6818  0.5174
Decision Tree         0.8361  0.6220     0.5244  0.8048  0.6350    0.7544  0.6722  0.5490
KNN                   0.8304  0.6226     0.6068  0.5695  0.5876    0.7878  0.6607  0.4989

Hyperparamètres retenus :
  Logistic Regression {'C': 10} | AUC CV = 0.846
  Decision Tree {'min_samples_leaf': 50, 'max_depth': 5} | AUC CV = 0.832
  KNN {'weights': 'uniform', 'n_neighbors': 41} | AUC CV = 0.833
  Random Forest {'n_estimators': 200, 'min_samples_leaf': 20, 'max_depth': 8} | AUC CV = 0.848
  XGBoost {'subsample':